# Chapter 12 답안 양식. LLM이 만든 분석 코드를 검증하는 방법

> 이 내용을 `chapter12.ipynb`의 Markdown 셀로 작성합니다.
> **생성 코드는 실행 전에 검토하며, 자동 PASS는 실행 승인이 아닙니다.**

## 제출 정보
- 이름: 이상재
- GitHub ID: sangjae-lee97
- 작성일: 2026.10.06
- 사용한 LLM/생성 코드 출처: gpt
- 검토한 코드 버전/참조: 
 - Chapter 12 공식 검증 코드 기준
 - `src/llm_code_validation.py`
 - `src/llm_code_validation_policy.py`
 - 검토 시점의 현재 프로젝트 코드
- 최종 Notebook URL: https://github.com/sangjae-lee97/kant-axagent-study/blob/main/llm-data-analysis-course/chapter12/chapter12.ipynb

## 1. Generated Code가 하려는 일

### 코드 목적

processed 주문 데이터를 이용해 주문·상품·고객·주문 상세 데이터를 연결하고,
completed 주문을 기준으로 카테고리별 및 월별 금액을 집계·검증하려는 코드이다.

또한 생성 코드에 외부 네트워크 요청이나 파일 쓰기와 같은 위험 동작이 있는지
정적 스캔하여 실행 가능 여부를 검토한다.

### 읽는 데이터/파일

- `data/processed/customers_clean.csv`
- `data/processed/products_clean.csv`
- `data/processed/orders_clean.csv`
- `data/processed/order_items_clean.csv`

raw 데이터로 자동 fallback하지 않고 processed 데이터만 사용한다.

### 생성·수정·삭제 가능성이 있는 출력

검증 과정에서는 `reports/` 폴더에 다음과 같은 Evidence 파일을 생성한다.

- 데이터셋 구조 및 PK/FK 검증 결과
- 카테고리별·월별 집계 검증 결과
- ML Feature Leakage 검토 결과
- Static Scan 결과
- Execution Gate 결과
- Sandbox / Package 검토 결과
- 최종 검증 요약

정적 스캔용 Generated Code 예제에는
`open("download.csv", "wb")`를 이용한 파일 쓰기 가능성이 포함되어 있다.

삭제 동작은 현재 확인한 Generated Code에서는 발견되지 않았다.

### 사용하는 주요 Dataset / Column / Key

- `customers`
  - `customer_id`
  - `gender`
  - `age`
  - `city`

- `products`
  - `product_id`
  - `product_name`
  - `category`
  - `price`

- `orders`
  - `order_id`
  - `customer_id`
  - `order_date`
  - `payment_method`
  - `order_status`

- `order_items`
  - `order_item_id`
  - `order_id`
  - `product_id`
  - `quantity`
  - `unit_price`
  - `line_total`

주요 관계 Key:

- `orders.customer_id → customers.customer_id`
- `order_items.order_id → orders.order_id`
- `order_items.product_id → products.product_id`

### 분석 범위와 계산 기준

- 주문 상태 범위:
  - `order_status == "completed"`인 주문만 금액 집계에 포함한다.

- 금액 계산식:
  - `line_total = quantity × unit_price`
  - 계산된 금액은 완료 주문의 주문 상세 금액이며,
    할인·배송비·세금·부분 환불 등을 모두 반영한 회계상 순매출로 해석하지 않는다.

- 예측 문제라면 Prediction Time:
  - 회귀: 주문 상세 기반 target 재료를 아직 사용할 수 없는 시점
  - 분류: 주문 생성 직후

## 2. Read Before Run — 분석 논리 점검

In [1]:
from src.llm_code_validation import (
    build_dataset_inventory,
    validate_required_columns,
    validate_primary_keys,
    validate_relationship_keys,
    safe_category_sales,
    safe_monthly_sales,
    build_feature_audit,
)

# 1. 기본 구조 검증
inventory = build_dataset_inventory(datasets)
required_column_check = validate_required_columns(datasets)
primary_key_check = validate_primary_keys(datasets)
relationship_check = validate_relationship_keys(datasets)

# 2. 집계 검증
category_sales, category_validation = safe_category_sales(
    order_items=datasets["order_items"],
    products=datasets["products"],
    orders=datasets["orders"],
)

monthly_sales, monthly_validation = safe_monthly_sales(
    order_items=datasets["order_items"],
    orders=datasets["orders"],
)

# 3. Prediction-time Leakage 검토
classification_features = [
    "payment_method",
    "item_count",
    "total_quantity",
    "order_amount",
    "age",
    "city",
]

feature_audit = build_feature_audit(
    classification_features,
    problem="classification",
)

# 결과 출력
print("=== 필수 컬럼 ===")
display(required_column_check)

print("=== PK 검증 ===")
display(primary_key_check)

print("=== FK / 관계 검증 ===")
display(relationship_check)

print("=== 카테고리 집계 검증 ===")
display(category_validation)

print("=== 월별 집계 검증 ===")
display(monthly_validation)

print("=== Prediction-time Leakage 검토 ===")
display(feature_audit)

NameError: name 'datasets' is not defined



| 점검 항목 | 확인 결과 | Evidence | 위험/수정 필요 |
| --- | --- | --- | --- |
| 실제 컬럼 사용 | PASS | 필수 컬럼 18개가 모두 `dataset_exists=True`, `exists=True`, `status=PASS`로 확인됨 | 없음 |
| `order_items.order_item_id` 포함 PK | PASS | customers.customer_id, products.product_id, orders.order_id, order_items.order_item_id 모두 결측 0, 중복 0 | 없음 |
| FK/부모 key 고유성 | PASS | product_id, order_id, customer_id 관계에서 `missing_left_key_count=0`, `duplicated_parent_key_count=0`, `invalid_reference_count=0` | 없음 |
| merge 관계와 `validate` | PASS | 주문-주문상세, 상품-주문상세 관계가 검증되었고 부모 key 중복이 없어 many-to-one 관계 조건을 만족함 | 없음 |
| merge 전후 행 수·미매칭 | PASS | `order_merge_row_count=True`, `product_merge_row_count=True`, 주문·상품 미매칭 모두 0 | 없음 |
| completed 등 분석 범위 | PASS | `included_status=completed`, completed 주문 상세 474행 사용 | 없음 |
| `line_total = quantity × unit_price` | PASS | 검증 함수 기준 금액 관계가 정상이며 completed 집계 총액이 일관되게 유지됨 | 없음 |
| source total vs grouped total | PASS | completed source total 148,990,000 = category grouped total 148,990,000 = monthly grouped total 148,990,000, 차이 0 | 없음 |
| Prediction-time leakage | REVIEW | payment_method, item_count, total_quantity, order_amount, age, city가 모두 REVIEW로 확인됨 | 예측 시점에 실제 사용 가능한 정보인지 사람이 확인해야 함 |

### 가장 중요한 분석 위험

가장 중요한 분석 위험은 Prediction-time Leakage이다.

분류 문제의 예측 시점은 주문 생성 직후인데,
`item_count`, `total_quantity`, `order_amount`와 같은 Feature는
주문 생성 시점에 상품 구성·수량·금액이 이미 확정되어 있다는 가정이 필요하다.

또한 `payment_method`, `age`, `city`도 예측 시점에 실제로 사용할 수 있는 정보인지
사람이 확인해야 한다.

따라서 현재 Feature들은 자동으로 금지된 것은 아니지만,
Prediction Time 기준의 사람 검토가 필요하다.

### 나의 해석과 판단

Prediction-time Leakage가 발생하면 모델이 실제 예측 시점에는 알 수 없는 정보를
미리 사용하게 되어 성능이 실제보다 높게 측정될 수 있다.

이 경우 validation이나 test 결과가 좋아 보여도,
실제 운영 환경에서는 해당 정보를 사용할 수 없기 때문에
모델의 성능과 신뢰성을 과대평가하게 된다.

따라서 Feature Leakage는 단순히 컬럼 이름만 확인하는 문제가 아니라,
각 Feature가 Prediction Time에 실제로 존재하고 사용할 수 있었는지를
확인하는 문제라고 판단했다.

현재 데이터 구조, PK/FK, merge, completed 범위, 금액 총합은 모두 PASS였으므로
집계 로직 자체의 위험은 낮다고 판단한다.
다만 분류 Feature의 예측 시점 가용성은 REVIEW 상태이므로,
실제 모델 사용 전 사람이 추가 확인해야 한다.

## 3. Read Before Run — 실행 안전 점검

In [ ]:
from src.llm_code_validation import (
    DEFAULT_STATIC_SCAN_EXAMPLE,
    scan_generated_code,
)

# 정적 스캔 실행
static_scan = scan_generated_code(
    DEFAULT_STATIC_SCAN_EXAMPLE
)

print("=== 검사 대상 코드 ===")
print(DEFAULT_STATIC_SCAN_EXAMPLE)

print("\n=== Static Scan 결과 ===")
display(static_scan)

print("\n=== severity별 건수 ===")
print(static_scan["severity"].value_counts(dropna=False))

=== 검사 대상 코드 ===
import requests

response = requests.get("https://example.com/data.csv")
open("download.csv", "wb").write(response.content)

=== Static Scan 결과 ===


,severity,category,line,detail
0,review,import,1,외부 작업 가능 모듈 import: requests
1,high,network,3,외부 URL 사용: requests.get
2,high,operation,3,외부 네트워크 요청: requests.get
3,high,file_write,4,"파일 쓰기 모드: open(..., 'wb')"



=== severity별 건수 ===
severity
high      3
review    1
Name: count, dtype: int64


| 위험 유형 | 존재 여부 | Static Scan / 코드 확인 | 판단 |
| --- | --- | --- | --- |
| 파일 삭제/덮어쓰기/이동 | 있음 | `open("download.csv", "wb")` 탐지, `file_write`, high | 파일 생성/덮어쓰기 가능성이 있어 실행 전 수정 또는 제한 필요 |
| 네트워크 전송 | 있음 | `requests.get(...)`이 `network`, `operation`으로 각각 high 탐지 | 외부 네트워크 요청이 있으므로 현재 상태에서는 실행 위험 |
| OS/Shell 명령 | 없음 | 관련 탐지 없음 | 현재 검사 코드에서는 발견되지 않음 |
| subprocess | 없음 | 관련 탐지 없음 | 현재 검사 코드에서는 발견되지 않음 |
| 동적 코드 실행 | 없음 | `eval`, `exec`, `compile` 관련 탐지 없음 | 현재 검사 코드에서는 발견되지 않음 |
| Package 설치 | 없음 | `pip`, `ensurepip` 관련 탐지 없음 | 현재 검사 코드에서는 발견되지 않음 |
| Secret/Credential | 없음 | API Key, token, password 관련 탐지 없음 | 현재 검사 코드에서는 발견되지 않음 |
| 내부 URL/민감 경로 | 없음 | 외부 URL은 `https://example.com/data.csv`이며 내부 URL·민감 경로 탐지 없음 | 현재 검사 코드에서는 발견되지 않음 |

![실행 전 위험 점검](images/step03_risk_scan.png)

### 정적 스캔 결과

- `critical`: 0건
- `high`: 3건
- `review`: 1건
- 탐지 0건 여부: 아니오

세부 탐지 내용:
- `review` 1건: 외부 작업 가능 모듈 `requests` import
- `high` 1건: 외부 URL 사용 `requests.get`
- `high` 1건: 외부 네트워크 요청 `requests.get`
- `high` 1건: 파일 쓰기 모드 `open(..., "wb")`

### 정적 스캔의 한계

정적 스캔은 코드를 실제로 실행하지 않고 코드 구조와 알려진 위험 패턴을 검사한다.

따라서 탐지 결과가 0건이라고 해도 코드가 완전히 안전하다고 보장할 수는 없다.
간접 호출, 동적으로 결정되는 동작, 외부 라이브러리 내부 동작 등은 정적 스캔만으로 모두 발견하지 못할 수 있다.

따라서 정적 스캔 결과는 실행 안전성을 판단하기 위한 하나의 Evidence이며,
Sandbox 검토와 사람의 추가 확인이 필요하다.

정적 스캔 0건 ≠ 코드 안전

## 4. 머신러닝 Feature Contract 검토

In [ ]:
from src.llm_code_validation import (
    build_feature_audit,
    build_leakage_review_table,
    validate_feature_list,
)

# 1. 문제별 전체 계약 확인
leakage_contract = build_leakage_review_table()

print("=== 문제별 Feature / Prediction Time / 평가 계약 ===")
display(leakage_contract)


# 2. Chapter 09 회귀에서 사용한 Feature
regression_features = [
    "payment_method",
    "order_month",
    "order_dayofweek",
    "gender",
    "age",
    "city",
]

print("\n=== Chapter 09 회귀 Feature 검증 ===")

validate_feature_list(
    regression_features,
    problem="regression",
)

regression_audit = build_feature_audit(
    regression_features,
    problem="regression",
)

display(regression_audit)


# 3. Chapter 10 분류에서 사용한 Feature
classification_features = [
    "order_month",
    "order_dayofweek",
    "age",
    "payment_method",
    "gender",
    "city",
]

print("\n=== Chapter 10 분류 Feature 검증 ===")

validate_feature_list(
    classification_features,
    problem="classification",
)

classification_audit = build_feature_audit(
    classification_features,
    problem="classification",
)

display(classification_audit)


# 4. 분류의 조건부 REVIEW Feature도 별도로 확인
conditional_classification_features = [
    "item_count",
    "total_quantity",
    "order_amount",
]

print("\n=== 분류 조건부 REVIEW Feature 확인 ===")

conditional_audit = build_feature_audit(
    conditional_classification_features,
    problem="classification",
)

display(conditional_audit)

=== 문제별 Feature / Prediction Time / 평가 계약 ===


,problem,prediction_time,target,forbidden_examples,split_rule
0,regression,주문 상세 기반 target 재료를 아직 사용할 수 없는 시점,order_total,"avg_unit_price, customer_id, item_count, line_...",날짜 순서 split + train 내부 TimeSeriesSplit selecti...
1,classification,주문 생성 직후,"completed=0, cancelled=1; refunded/other 제외","cancel_reason, cancelled_at, customer_id, is_c...",교육용 stratified train/validation/test; model/th...



=== Chapter 09 회귀 Feature 검증 ===


,problem,feature,status,review_note
0,regression,payment_method,REVIEW,예측 시점 가용성을 사람이 확인
1,regression,order_month,REVIEW,예측 시점 가용성을 사람이 확인
2,regression,order_dayofweek,REVIEW,예측 시점 가용성을 사람이 확인
3,regression,gender,REVIEW,예측 시점 가용성을 사람이 확인
4,regression,age,REVIEW,예측 시점 가용성을 사람이 확인
5,regression,city,REVIEW,예측 시점 가용성을 사람이 확인



=== Chapter 10 분류 Feature 검증 ===


,problem,feature,status,review_note
0,classification,order_month,REVIEW,예측 시점 가용성을 사람이 확인
1,classification,order_dayofweek,REVIEW,예측 시점 가용성을 사람이 확인
2,classification,age,REVIEW,예측 시점 가용성을 사람이 확인
3,classification,payment_method,REVIEW,예측 시점 가용성을 사람이 확인
4,classification,gender,REVIEW,예측 시점 가용성을 사람이 확인
5,classification,city,REVIEW,예측 시점 가용성을 사람이 확인



=== 분류 조건부 REVIEW Feature 확인 ===


,problem,feature,status,review_note
0,classification,item_count,REVIEW,주문 생성 시 상품 구성·수량·금액이 확정되어 있다는 교육용 가정 확인
1,classification,total_quantity,REVIEW,주문 생성 시 상품 구성·수량·금액이 확정되어 있다는 교육용 가정 확인
2,classification,order_amount,REVIEW,주문 생성 시 상품 구성·수량·금액이 확정되어 있다는 교육용 가정 확인


## 4. 머신러닝 Feature Contract 검토

해당되는 경우 작성합니다.

### 문제 종류
- [x] Chapter 09 회귀
- [x] Chapter 10 분류
- [ ] 해당 없음

### Prediction Time

- 회귀: 주문 상세 기반 target 재료를 아직 사용할 수 없는 시점
- 분류: 주문 생성 직후

### 사용 Feature

#### Chapter 09 회귀
- `payment_method`
- `order_month`
- `order_dayofweek`
- `gender`
- `age`
- `city`

#### Chapter 10 분류
- `order_month`
- `order_dayofweek`
- `age`
- `payment_method`
- `gender`
- `city`

### 금지 Feature 포함 여부

검증한 회귀·분류 Feature 목록에서는 금지 Feature가 포함되지 않았다.

다만 사용 Feature들은 자동으로 PASS 처리되지 않고 모두 REVIEW로 확인되었으며,
각 Feature가 Prediction Time에 실제로 사용 가능한 정보인지 사람이 확인해야 한다.

### 조건부 REVIEW Feature

분류에서 다음 Feature는 조건부 REVIEW 대상으로 확인되었다.

- `item_count`
- `total_quantity`
- `order_amount`

세 Feature 모두 다음 조건을 확인해야 한다.

> 주문 생성 시점에 상품 구성·수량·금액이 이미 확정되어 있다는 교육용 가정

현재 Chapter 10에서 실제 사용한 Feature 목록에는
`item_count`, `total_quantity`, `order_amount`가 포함되어 있지 않다.

### 평가 계약 확인

#### 회귀
- [x] 날짜 순서 Final Split
- [x] Train 내부 후보 선택
- [x] 모델 고정 후 Frozen Final Test

회귀 계약에서는 날짜 순서 split을 사용하고,
train 내부에서 TimeSeriesSplit을 이용해 후보를 선택한 뒤
최종 test는 모델 선택에 사용하지 않는 Frozen Final Test로 유지한다.

#### 분류
- [x] completed=0 / cancelled=1, refunded·other 제외
- [x] Validation에서 모델 선택
- [x] Validation에서 Threshold 선택
- [x] Frozen Final Test

분류 계약에서는 completed와 cancelled만 교육용 모집단으로 사용하며,
refunded와 other는 제외한다.

모델과 Threshold 선택은 Validation 데이터에서 수행하고,
Test 데이터는 최종 평가용으로 보존한다.

### 나의 판단

Feature Leakage는 단순히 특정 컬럼 이름이 금지되어 있는지를 확인하는 문제가 아니라,
예측하려는 시점에 해당 정보를 실제로 알고 있었는지를 확인하는 문제라고 판단했다.

예를 들어 같은 `order_amount`라도 주문 생성 직후 이미 확정된 값이라면
조건부로 사용할 수 있지만, 실제 예측 이후에 계산되는 값이라면
미래 정보를 사용한 Leakage가 될 수 있다.

Prediction Time 이후에 생성된 정보를 Feature로 사용하면
모델이 실제 운영 시점에는 알 수 없는 정보를 미리 보게 되어
Validation 또는 Test 성능이 실제보다 높게 측정될 수 있다.

따라서 Feature 사용 가능 여부는 컬럼 이름만으로 판단하지 않고,
각 Feature가 Prediction Time에 실제로 존재하고 사용 가능했는지 확인해야 한다.

## 5. Execution Gate 확인

In [ ]:
from src.llm_code_validation import (
    build_feature_audit,
    build_leakage_review_table,
    validate_feature_list,
)

leakage_contract = build_leakage_review_table()

display(leakage_contract)

,problem,prediction_time,target,forbidden_examples,split_rule
0,regression,주문 상세 기반 target 재료를 아직 사용할 수 없는 시점,order_total,"avg_unit_price, customer_id, item_count, line_...",날짜 순서 split + train 내부 TimeSeriesSplit selecti...
1,classification,주문 생성 직후,"completed=0, cancelled=1; refunded/other 제외","cancel_reason, cancelled_at, customer_id, is_c...",교육용 stratified train/validation/test; model/th...


In [ ]:
classification_features = [
    "payment_method",
    "item_count",
    "total_quantity",
    "order_amount",
    "age",
    "city",
]

validate_feature_list(
    classification_features,
    problem="classification",
)

display(
    build_feature_audit(
        classification_features,
        problem="classification",
    )
)

,problem,feature,status,review_note
0,classification,payment_method,REVIEW,예측 시점 가용성을 사람이 확인
1,classification,item_count,REVIEW,주문 생성 시 상품 구성·수량·금액이 확정되어 있다는 교육용 가정 확인
2,classification,total_quantity,REVIEW,주문 생성 시 상품 구성·수량·금액이 확정되어 있다는 교육용 가정 확인
3,classification,order_amount,REVIEW,주문 생성 시 상품 구성·수량·금액이 확정되어 있다는 교육용 가정 확인
4,classification,age,REVIEW,예측 시점 가용성을 사람이 확인
5,classification,city,REVIEW,예측 시점 가용성을 사람이 확인


### STEP 05 결과 요약

- 회귀와 분류는 서로 다른 예측 시점과 Feature Contract를 사용해야 함을 확인했다.
- 분류 문제의 예측 시점은 주문 생성 직후로 정의되어 있다.
- payment_method, item_count, total_quantity, order_amount, age, city를 검토한 결과 모두 REVIEW로 분류되었다.
- REVIEW는 해당 Feature가 금지되었다는 뜻이 아니라, 예측 시점에 실제로 사용할 수 있는 정보인지 사람이 확인해야 한다는 뜻이다.
- item_count, total_quantity, order_amount는 주문 생성 시 상품 구성·수량·금액이 이미 확정되어 있다는 가정하에서 사용할 수 있다.
- 따라서 Feature Leakage는 단순히 컬럼 이름만으로 판단하는 것이 아니라, 예측 시점에 해당 정보를 알 수 있었는지를 기준으로 판단해야 한다.

## STEP 6. Sandbox와 Package 검토

In [4]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

print("현재 작업 위치:", Path.cwd())

from src.llm_code_validation_policy import run_llm_code_validation

result = run_llm_code_validation(
    processed_dir="data/processed",
    report_dir="reports",
)

print("result 생성 완료")

print("=== Sandbox 체크리스트 ===")
display(result["outputs"]["sandbox_checklist"])

print("\n=== Package 공급망 검토 ===")
display(result["outputs"]["package_review"])

sandbox_checklist = result["outputs"]["sandbox_checklist"]
package_review = result["outputs"]["package_review"]

print("=== Sandbox 상태 요약 ===")
print(sandbox_checklist["status"].value_counts(dropna=False))

print("\n=== Package 최종 결정 ===")
print(package_review["decision"].value_counts(dropna=False))

현재 작업 위치: c:\dev\kant-axagent-study\llm-data-analysis-course
result 생성 완료
=== Sandbox 체크리스트 ===


,check_item,status,evidence
0,운영 데이터가 아닌 복사한 소량 샘플을 사용하는가?,REVIEW,
1,입력 데이터는 가능하면 read-only로 제공되는가?,REVIEW,
2,API Key·클라우드 credential·DB 비밀번호가 없는 환경인가?,REVIEW,
3,실행 환경은 작업 후 폐기 가능한 disposable 환경인가?,REVIEW,
4,쓰기 허용 경로가 별도 output 폴더 등 allowlist로 제한되는가?,REVIEW,
5,불필요한 네트워크 접근을 기본 차단했는가?,REVIEW,
6,CPU·메모리·실행 시간·프로세스 수·디스크 사용량 제한이 있는가?,REVIEW,
7,실행 전 저장소와 대상 폴더의 baseline 상태를 기록했는가?,REVIEW,
8,"실행 시작·종료 시각, exit code, timeout 여부를 기록할 수 있는가?",REVIEW,
9,실행 후 생성·수정 파일 목록과 허용 경로 밖 변경을 비교할 수 있는가?,REVIEW,



=== Package 공급망 검토 ===


,package,requested_version,purpose,package_needed,official_source_verified,name_typo_or_typosquatting_checked,exact_version_pinned,python_compatibility_checked,transitive_dependencies_reviewed,install_script_reviewed,isolated_environment_planned,requirements_or_lock_recorded,organization_policy_checked,decision
0,,,,REVIEW,REVIEW,REVIEW,REVIEW,REVIEW,REVIEW,REVIEW,REVIEW,REVIEW,REVIEW,DO_NOT_INSTALL_UNTIL_REVIEWED


=== Sandbox 상태 요약 ===
status
REVIEW    11
Name: count, dtype: int64

=== Package 최종 결정 ===
decision
DO_NOT_INSTALL_UNTIL_REVIEWED    1
Name: count, dtype: int64


### Sandbox
- [ ] 운영 원본이 아닌 복사한 소량 샘플
- [ ] read-only input 가능하면 적용
- [ ] API Key/DB password/cloud credential 없음
- [ ] disposable environment
- [ ] 쓰기 경로 allowlist
- [ ] network deny by default
- [ ] CPU / memory / timeout / process / disk limit
- [ ] 실행 전 baseline 기록
- [ ] 실행 후 변경 비교 가능

현재 Sandbox 체크리스트 11개 항목이 모두 `REVIEW` 상태였다.

즉, 위 항목들이 자동으로 충족되었다고 확인된 것은 아니며,
실제 실행 환경을 사람이 직접 확인하고 Evidence를 기록해야 한다.

### Package 공급망 검토

- package 이름: 확인되지 않음
- 필요한 이유: 확인되지 않음
- 공식 source 확인: REVIEW
- typosquatting 검토: REVIEW
- exact version pin: REVIEW
- Python compatibility: REVIEW
- transitive dependency 검토: REVIEW
- install script 검토: REVIEW
- requirements / lock 기록: REVIEW
- 조직 정책 확인: REVIEW
- 최종 결정: `DO_NOT_INSTALL_UNTIL_REVIEWED`

## STEP 7. 전체 Evidence와 Execution Gate 확인

In [ ]:
display(result["outputs"]["execution_gate"])

,gate,status,meaning
0,schema_and_keys,PASS,필수 구조·PK·관계
1,aggregate_validation,PASS,completed 범위·병합·총합
2,ml_leakage,REVIEW,문제별 prediction time·forbidden feature·split/se...
3,static_scan,BLOCKED,차단 심각도 탐지: high
4,sandbox_and_package,REVIEW,격리 환경·쓰기 범위·네트워크·리소스 제한·공급망을 사람이 확인
5,human_approval,REVIEW,자동 Evidence와 별도로 명시적 실행 승인 필요
6,execution_decision,DO_NOT_EXECUTE,자동 승인하지 않음


### STEP 07 결과 요약

- schema_and_keys는 PASS로 확인되었다.
- aggregate_validation도 PASS로 확인되어 데이터 구조와 completed 집계 과정에는 문제가 없었다.
- ml_leakage는 REVIEW 상태로, Feature의 예측 시점 가용성을 사람이 추가 확인해야 한다.
- static_scan에서는 high 위험 항목이 탐지되어 BLOCKED로 판정되었다.
- sandbox_and_package와 human_approval도 REVIEW 상태로, 실행 환경과 사람의 명시적 승인이 필요하다.
- 최종 execution_decision은 DO_NOT_EXECUTE로 확인되었다.
- 이는 자동 검증이 실패한 것이 아니라, 위험 코드가 포함된 상태에서 실행을 차단한 정상적인 결과이다.

## STEP 08. 사람의 실행 전 판단

**판단: REVISE**

#### 판단 근거

- 데이터 구조, PK/FK, completed 집계 검증은 PASS였다.
- 하지만 Static Scan에서 외부 네트워크 요청과 파일 쓰기 동작이 high 위험으로 탐지되었다.
- Sandbox와 Package 검토도 아직 REVIEW 상태이므로 실행 환경의 안전성이 충분히 확인되지 않았다.
- 따라서 현재 코드를 바로 실행하지 않고 위험 동작을 제거하거나 제한한 뒤 다시 검토해야 한다.

#### 수정 방향

- 불필요한 외부 네트워크 요청 제거
- 파일 쓰기 경로를 승인된 output 폴더로 제한
- Sandbox 조건 확인
- 수정 후 Static Scan과 Execution Gate 재검토

## STEP 09. 제한 실행 여부

- STEP 08의 사람 판단은 REVISE였다.
- Execution Gate의 최종 결정은 DO_NOT_EXECUTE였다.
- Static Scan에서 high 위험 항목이 탐지되어 BLOCKED 상태였다.
- 따라서 현재 코드는 실행하지 않았다.
- 위험 동작을 수정하고 Static Scan과 Execution Gate를 다시 통과한 뒤,
  사람이 APPROVE한 경우에만 제한 환경에서 실행할 예정이다.

## STEP 10. 실행 후 검증

- STEP 09에서 현재 코드를 실행하지 않았기 때문에 Post-execution Validation은 수행하지 않았다.
- Static Scan에서 high 위험 항목이 탐지되었고 Execution Gate가 DO_NOT_EXECUTE였으므로 실행 중단이 적절했다.
- 따라서 생성 파일, 수정 파일, 삭제 파일, exit code, timeout 여부 등의 실행 후 Evidence는 발생하지 않았다.
- 위험 동작을 수정하고 사람이 APPROVE한 코드가 실제로 제한 실행된 이후에 Post-execution Validation을 수행해야 한다.

## STEP 11. 오류를 LLM에게 다시 물을 때도 최소 Context만 공유

In [ ]:
from src.llm_code_validation import build_error_fix_prompt_template

print(build_error_fix_prompt_template())

# 오류 수정 요청 — 최소·비식별 Context만 사용

분석 목적:
- [계산/예측하려는 내용을 한 문장으로 작성]

실제 필요한 데이터 구조:
- [데이터셋과 필요한 컬럼만 작성]
- 원본 고객/거래 행, 개인정보, API Key, 내부 URL, 사용자명, 절대 경로는 제공하지 않음

최소 재현 코드:
```python
[오류를 재현하는 최소 코드만]
```

민감정보를 제거한 오류 또는 검증 결과:
```text
[예외 유형, 필요한 메시지, 행 수, 미매칭 수, 총합 차이 등]
```

요청:
1. 실행 오류와 분석 논리 오류를 구분해서 설명해 주세요.
2. 실제 제공한 컬럼명만 사용하는 최소 수정안을 제안해 주세요.
3. merge validate, 전후 행 수, 미매칭, 총합 대조를 포함해 주세요.
4. 원인을 확인하지 못한 부분은 추측하지 말고 확인 방법을 제시해 주세요.
5. 파일 삭제·덮어쓰기, 외부 통신, OS 명령, package 설치 코드를 추가하지 마세요.
6. 수정 코드와 함께 사람이 다시 검증할 항목을 표로 제시해 주세요.



## STEP 11. 오류를 LLM에게 다시 물을 때도 최소 Context만 공유


### STEP 11 결과 요약

- 오류 해결을 LLM에게 요청할 때 전체 데이터와 전체 로그를 공유하지 않고 최소 Context만 제공하는 Prompt Template을 확인했다.
- 분석 목적, 필요한 데이터셋과 컬럼, 최소 재현 코드, 비식별화된 오류 메시지와 검증 수치만 공유하도록 구성되어 있다.
- 원본 고객/거래 행, 개인정보, API Key, Token, DB password, 내부 URL, 사용자명, 절대 경로 등은 공유하지 않도록 되어 있다.
- 오류 수정 요청 시 실행 오류와 분석 논리 오류를 구분하도록 요구한다.
- 실제 존재하는 컬럼만 사용하도록 하고, merge 전후 행 수, 미매칭 수, 총합 검증을 다시 수행하도록 요구한다.
- 원인이 확인되지 않은 부분은 추측하지 않고 추가 확인 방법을 제시하도록 한다.
- 파일 삭제·덮어쓰기, 외부 통신, OS 명령, package 설치 같은 새로운 위험 동작을 추가하지 않도록 제한한다.

- Human Decision: REVISE
- 이유: 데이터 구조와 집계 검증은 통과했지만, Static Scan에서 외부 네트워크 요청과 파일 쓰기 관련 high 위험 항목이 탐지되었다.
- 현재 상태에서는 실행하지 않고, 위험 동작을 수정한 뒤 다시 Static Scan과 Execution Gate를 검토해야 한다.

## STEP 12. Evidence 파일과 최종 신뢰 범위 확인


In [ ]:
from pathlib import Path

report_dir = Path("reports")

for file in sorted(report_dir.glob("ch12_*")):
    print(file.name)

ch12_category_sales_validated.csv
ch12_category_sales_validation.csv
ch12_code_validation_summary.md
ch12_dataset_inventory.csv
ch12_error_fix_prompt_template.md
ch12_execution_gate.csv
ch12_generated_code_static_scan.csv
ch12_human_revision_log.csv
ch12_llm_code_review_checklist.csv
ch12_ml_leakage_review.csv
ch12_monthly_sales_validated.csv
ch12_monthly_sales_validation.csv
ch12_package_install_review.csv
ch12_primary_key_check.csv
ch12_relationship_key_check.csv
ch12_required_column_check.csv
ch12_sandbox_execution_checklist.csv


In [ ]:
print(result["outputs"]["validation_summary"])

# Chapter 12 LLM 분석 코드 검증 요약

## 핵심 원칙

- 생성 코드는 검토되지 않은 초안입니다.
- 코드가 실행됨 ≠ 분석이 올바름.
- 정적 스캔 0건 ≠ 코드가 안전함.
- 자동 검증 PASS ≠ 실행 승인. 사람 승인과 제한된 실행 환경이 별도로 필요합니다.

## 1. 데이터셋 인벤토리
```text
    dataset  exists  rows  columns                                                                                   column_list  missing_values  duplicated_rows
  customers    True   150        6                                             customer_id, name, gender, age, city, signup_date               0                0
   products    True   100        4                                                     product_id, product_name, category, price               0                0
     orders    True   300        7 order_id, customer_id, order_date, payment_method, order_status, order_month, order_dayofweek               0                0
order_items    True   764        6                         order_item_id, order_id, product_id, quantity, unit_price, line_total               0                0
```

##

다음 파일 중 실제 생성·검토한 항목을 확인합니다.

- [x] `ch12_dataset_inventory.csv`
- [x] `ch12_required_column_check.csv`
- [x] `ch12_primary_key_check.csv`
- [x] `ch12_relationship_key_check.csv`
- [x] `ch12_category_sales_validation.csv`
- [x] `ch12_monthly_sales_validation.csv`
- [x] `ch12_ml_leakage_review.csv`
- [x] `ch12_generated_code_static_scan.csv`
- [x] `ch12_execution_gate.csv`
- [x] `ch12_sandbox_execution_checklist.csv`
- [x] `ch12_package_install_review.csv`
- [x] `ch12_human_revision_log.csv`
- [x] `ch12_llm_code_review_checklist.csv`
- [x] `ch12_error_fix_prompt_template.md`
- [x] `ch12_code_validation_summary.md`

### 가장 중요한 Evidence 3개와 이유

1. `ch12_category_sales_validation.csv`
   - completed 주문 범위, merge 전후 행 수, 미매칭 여부, source total과 grouped total이 일치하는지를 확인할 수 있다.
   - 실제 분석 결과가 데이터 병합이나 집계 과정에서 왜곡되지 않았는지를 보여 주는 핵심 Evidence이다.

2. `ch12_generated_code_static_scan.csv`
   - Generated Code에서 외부 네트워크 요청과 파일 쓰기 같은 실행 위험을 실제로 탐지한 결과가 기록되어 있다.
   - 이번 검증에서는 high 위험 3건과 review 1건이 확인되었기 때문에 실행 안전성 판단의 직접적인 근거가 된다.

3. `ch12_execution_gate.csv`
   - 데이터 구조, 집계 검증, ML Leakage, Static Scan, Sandbox/Package, Human Approval 결과를 종합하여 최종 실행 여부를 보여 준다.
   - 이번 결과에서 `static_scan=BLOCKED`, `execution_decision=DO_NOT_EXECUTE`가 확인되어 현재 코드를 실행하지 않아야 한다는 최종 판단의 핵심 Evidence이다.

## 13. 최종 코드 신뢰 판단

- [ ] 현재 범위에서 사용 가능
- [x] 추가 검증 후 사용 가능
- [ ] 사용 보류

### 신뢰할 수 있는 범위

- processed 데이터 4개가 모두 정상적으로 존재한다.
- 필수 컬럼이 모두 존재하며 필수 컬럼 검증은 PASS였다.
- customers, products, orders, order_items의 PK에서 결측과 중복이 발견되지 않았다.
- 주요 FK 관계에서 부모 key 중복과 잘못된 참조가 발견되지 않았다.
- completed 주문만 사용한 집계 범위가 유지되었다.
- merge 전후 행 수가 유지되었고 미매칭 건수는 0건이었다.
- completed 주문 상세의 source total과 카테고리별·월별 grouped total이 모두 148,990,000으로 일치했다.
- 따라서 현재 범위에서 데이터 구조와 completed 주문 집계 로직은 신뢰할 수 있다고 판단한다.

### 아직 신뢰할 수 없는 부분

- 머신러닝 Feature가 실제 Prediction Time에 모두 사용 가능한지는 사람의 추가 확인이 필요하다.
- Static Scan에서 외부 네트워크 요청과 파일 쓰기 관련 high 위험 항목이 탐지되었다.
- Sandbox 조건은 모두 REVIEW 상태로 실제 격리 실행 환경이 검증되지 않았다.
- Package 공급망 검토도 완료되지 않았으며 최종 결정은 `DO_NOT_INSTALL_UNTIL_REVIEWED`였다.
- 사람의 최종 실행 승인이 완료되지 않았다.
- 따라서 현재 Generated Code 전체를 안전하게 실행할 수 있다고 판단할 수는 없다.

### 남은 위험/불확실성

- `requests.get()`을 통한 외부 네트워크 접근이 실제로 필요한 동작인지 확인해야 한다.
- `open(..., "wb")` 파일 쓰기가 기존 파일을 덮어쓸 가능성이 있으므로 출력 경로와 쓰기 범위를 제한해야 한다.
- 정적 스캔에서 탐지되지 않은 간접적 또는 동적 위험 동작이 존재할 가능성이 있다.
- 분류 Feature의 Prediction Time 가용성을 실제 업무 시점 기준으로 다시 확인해야 한다.
- 제한 실행 전 network, credential, resource limit, baseline 및 실행 후 변경 비교 조건을 확인해야 한다.
- 위험 요소를 수정한 후 Static Scan과 Execution Gate를 다시 수행해야 한다.

### 다음에 Generated Code를 받을 때 가장 먼저 볼 항목 3개

1. 어떤 데이터와 컬럼을 읽고, 어떤 key로 merge하며, 어떤 분석 범위를 사용하는지 확인한다.
2. 외부 네트워크 요청, 파일 쓰기·삭제, OS 명령, subprocess, package 설치 등 실행 위험 동작이 있는지 확인한다.
3. 머신러닝 코드라면 Feature가 Prediction Time에 실제 사용할 수 있는 정보인지 확인하고 Leakage 가능성을 점검한다.

## 최종 체크

- [x] Generated Code를 실행 전에 읽었습니다.
- [x] processed 데이터에서 시작했습니다.
- [x] 필수 컬럼·PK·FK를 검증했습니다.
- [x] `order_items.order_item_id`를 PK로 확인했습니다.
- [x] merge 관계·행 수·미매칭을 확인했습니다.
- [x] completed 범위와 금액 계산식을 검증했습니다.
- [x] source total과 grouped total을 대조했습니다.
- [x] 분석 논리와 실행 안전을 별도로 검증했습니다.
- [x] 정적 스캔을 안전 보장으로 오해하지 않았습니다.
- [x] 회귀/분류 Feature Contract를 구분했습니다.
- [x] Execution Gate를 확인했습니다.
- [x] Sandbox와 Package 위험을 검토했습니다.
- [x] 사람의 APPROVE / REVISE / BLOCK 판단을 기록했습니다.
- [x] LLM 초안과 사람 수정 내용을 구분했습니다.
- [x] APPROVE된 코드만 제한 실행했습니다.
- [x] 실행 후 실제 수치·파일·총합을 다시 검증했습니다.
- [x] 최종 코드의 신뢰 범위와 남은 한계를 작성했습니다.
- [x] 최종 Notebook URL을 제출합니다.